# 8.2 Quantization Techniques — Apply

## Objective

Hands-on quantization of ONNX models. You will:

1. Build an FP32 model and apply **dynamic INT8 quantization**
2. Implement a `CalibrationDataReader` and apply **static INT8 quantization**
3. Compare FP32 vs INT8: accuracy, file size, and performance
4. Understand the quantization formula and error bounds

**Key formula — affine quantization:**

$$x_q = \text{clamp}\!\left(\left\lfloor \frac{x}{s} \right\rceil + z,\; 0,\; 2^b - 1\right)$$

where $s$ is the scale, $z$ is the zero-point, and $b$ is the bit-width (8).

**Dequantization:**

$$\hat{x} = s \cdot (x_q - z)$$

**Quantization error bound:**

$$|x - \hat{x}| \leq \frac{s}{2} = \frac{x_{\max} - x_{\min}}{2(2^b - 1)}$$

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_dynamic, quantize_static, QuantType,
    CalibrationDataReader, CalibrationMethod, QuantFormat
)
import os
import time
import tempfile

TMPDIR = tempfile.mkdtemp(prefix='onnx_quant_')
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")
print(f"Temp dir: {TMPDIR}")

---
## Exercise 1: Build an FP32 Model for Quantization

We build a multi-layer perceptron: $y = \text{ReLU}(\text{ReLU}(XW_1 + b_1)W_2 + b_2)$

This gives enough structure for quantization to work with — multiple weight matrices
and activations that benefit from INT8 computation.

In [ ]:
np.random.seed(42)

N, D_in, D_hidden, D_out = 8, 64, 128, 32

W1 = np.random.randn(D_in, D_hidden).astype(np.float32) * 0.05
b1 = np.random.randn(D_hidden).astype(np.float32) * 0.01
W2 = np.random.randn(D_hidden, D_out).astype(np.float32) * 0.05
b2 = np.random.randn(D_out).astype(np.float32) * 0.01

nodes = [
    helper.make_node('MatMul', ['X', 'W1'], ['h1']),
    helper.make_node('Add', ['h1', 'b1'], ['h1b']),
    helper.make_node('Relu', ['h1b'], ['a1']),
    helper.make_node('MatMul', ['a1', 'W2'], ['h2']),
    helper.make_node('Add', ['h2', 'b2'], ['h2b']),
    helper.make_node('Relu', ['h2b'], ['Y']),
]

inits = [
    numpy_helper.from_array(W1, 'W1'), numpy_helper.from_array(b1, 'b1'),
    numpy_helper.from_array(W2, 'W2'), numpy_helper.from_array(b2, 'b2'),
]

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [N, D_in])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [N, D_out])

graph = helper.make_graph(nodes, 'mlp_fp32', [X_info], [Y_info], initializer=inits)
fp32_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(fp32_model)

fp32_path = os.path.join(TMPDIR, 'mlp_fp32.onnx')
onnx.save(fp32_model, fp32_path)

print(f"FP32 model: {len(fp32_model.graph.node)} nodes")
print(f"  Params: W1={W1.shape}, b1={b1.shape}, W2={W2.shape}, b2={b2.shape}")
total_params = W1.size + b1.size + W2.size + b2.size
print(f"  Total parameters: {total_params:,}")
print(f"  FP32 file size: {os.path.getsize(fp32_path):,} bytes")

# Run FP32 inference
sess_fp32 = ort.InferenceSession(fp32_path, providers=['CPUExecutionProvider'])
x_test = np.random.randn(N, D_in).astype(np.float32)
y_fp32 = sess_fp32.run(None, {'X': x_test})[0]
print(f"\nFP32 output shape: {y_fp32.shape}")
print(f"FP32 output range: [{y_fp32.min():.4f}, {y_fp32.max():.4f}]")

---
## Exercise 2: Dynamic INT8 Quantization

Dynamic quantization quantizes **weights** offline (stored as INT8) and quantizes
**activations** on-the-fly at inference. No calibration data is needed.

**When to use:** NLP models, models where activation ranges vary significantly per input.

The weight quantization scale is computed from the full weight range:

$$s_w = \frac{\max(|W|)}{2^{b-1} - 1}$$

In [ ]:
dynamic_path = os.path.join(TMPDIR, 'mlp_dynamic_int8.onnx')

quantize_dynamic(
    model_input=fp32_path,
    model_output=dynamic_path,
    weight_type=QuantType.QInt8,
)

dyn_model = onnx.load(dynamic_path)
print(f"Dynamic INT8 model: {len(dyn_model.graph.node)} nodes")
print(f"  Op types: {sorted(set(n.op_type for n in dyn_model.graph.node))}")
print(f"  File size: {os.path.getsize(dynamic_path):,} bytes")
print(f"  Size reduction: {100*(1 - os.path.getsize(dynamic_path)/os.path.getsize(fp32_path)):.1f}%")

# Run dynamic INT8 inference
sess_dyn = ort.InferenceSession(dynamic_path, providers=['CPUExecutionProvider'])
y_dyn = sess_dyn.run(None, {'X': x_test})[0]

# Compare outputs
max_diff = np.max(np.abs(y_fp32 - y_dyn))
mean_diff = np.mean(np.abs(y_fp32 - y_dyn))
rel_error = np.mean(np.abs(y_fp32 - y_dyn) / (np.abs(y_fp32) + 1e-8))

print(f"\nFP32 vs Dynamic INT8:")
print(f"  Max absolute diff:  {max_diff:.6f}")
print(f"  Mean absolute diff: {mean_diff:.6f}")
print(f"  Mean relative error: {rel_error:.4%}")

---
## Exercise 3: Static INT8 Quantization with Calibration

Static quantization determines activation scales **offline** using representative
calibration data, then embeds QuantizeLinear/DequantizeLinear (QDQ) nodes.

**When to use:** Vision models, any model where activation ranges are stable across inputs.

The calibration step collects activation statistics:

$$s_a = \frac{x_{\max}^{\text{calib}} - x_{\min}^{\text{calib}}}{2^b - 1}, \quad
z = \text{clamp}\!\left(\left\lfloor -\frac{x_{\min}^{\text{calib}}}{s_a} \right\rceil,\; 0,\; 2^b{-}1\right)$$

In [ ]:
class SyntheticCalibrationReader(CalibrationDataReader):
    """Provides calibration batches for static quantization."""

    def __init__(self, input_name, shape, n_batches=20, seed=0):
        self.input_name = input_name
        self.rng = np.random.default_rng(seed)
        self.batches = [
            self.rng.standard_normal(shape).astype(np.float32) * 0.5
            for _ in range(n_batches)
        ]
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.batches):
            return None
        batch = self.batches[self.idx]
        self.idx += 1
        return {self.input_name: batch}

static_path = os.path.join(TMPDIR, 'mlp_static_int8.onnx')

calib_reader = SyntheticCalibrationReader('X', (N, D_in), n_batches=20)

try:
    quantize_static(
        model_input=fp32_path,
        model_output=static_path,
        calibration_data_reader=calib_reader,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        calibrate_method=CalibrationMethod.MinMax,
    )

    stat_model = onnx.load(static_path)
    print(f"Static INT8 model: {len(stat_model.graph.node)} nodes")
    print(f"  QDQ ops: {sum(1 for n in stat_model.graph.node if 'Quantize' in n.op_type)}")
    print(f"  File size: {os.path.getsize(static_path):,} bytes")
    print(f"  Size reduction: {100*(1 - os.path.getsize(static_path)/os.path.getsize(fp32_path)):.1f}%")

    sess_stat = ort.InferenceSession(static_path, providers=['CPUExecutionProvider'])
    y_stat = sess_stat.run(None, {'X': x_test})[0]

    max_diff_stat = np.max(np.abs(y_fp32 - y_stat))
    print(f"\nFP32 vs Static INT8:")
    print(f"  Max absolute diff: {max_diff_stat:.6f}")
    print(f"  Mean relative error: {np.mean(np.abs(y_fp32 - y_stat)/(np.abs(y_fp32)+1e-8)):.4%}")
    HAS_STATIC = True
except Exception as e:
    print(f"Static quantization failed: {type(e).__name__}: {e}")
    print("This can happen with some ORT versions. Dynamic quantization still works.")
    HAS_STATIC = False

---
## Exercise 4: File Size Comparison

INT8 weights use $\frac{1}{4}$ the memory of FP32. For a model with $P$ parameters:

$$\text{Size}_{\text{FP32}} \approx 4P \text{ bytes}, \quad
\text{Size}_{\text{INT8}} \approx P + \text{overhead}_{\text{scales}} \text{ bytes}$$

The theoretical compression ratio is ~4x, minus overhead from scale/zero-point storage.

In [ ]:
models_info = [
    ('FP32 (original)', fp32_path),
    ('Dynamic INT8', dynamic_path),
]
if HAS_STATIC:
    models_info.append(('Static INT8 (QDQ)', static_path))

print(f"{'Model':<25} {'Size (bytes)':>12} {'Size (KB)':>10} {'Ratio':>8}")
print('-' * 60)
fp32_size = os.path.getsize(fp32_path)
for name, path in models_info:
    sz = os.path.getsize(path)
    ratio = sz / fp32_size
    print(f"{name:<25} {sz:>12,} {sz/1024:>10.1f} {ratio:>7.2f}x")

# Theoretical analysis
print(f"\nTheoretical analysis:")
print(f"  Total FP32 params: {total_params:,}")
print(f"  FP32 param bytes:  {total_params * 4:,} (4 bytes each)")
print(f"  INT8 param bytes:  {total_params * 1:,} (1 byte each)")
print(f"  Theoretical ratio: {total_params / (total_params * 4):.2f}x (plus scale overhead)")

---
## Exercise 5: Accuracy Impact Analysis

Quantization introduces error. We measure the distribution of errors across
many random inputs to assess the practical impact.

For symmetric quantization, the maximum per-element error is:

$$\epsilon_{\max} = \frac{s}{2} = \frac{\max(|W|)}{2(2^{b-1}-1)}$$

Error accumulates through layers, so deeper models show larger total error.

In [ ]:
n_samples = 200
diffs_dyn = []
diffs_stat = []

for i in range(n_samples):
    x_i = np.random.randn(N, D_in).astype(np.float32) * 0.5
    y_fp = sess_fp32.run(None, {'X': x_i})[0]
    y_dy = sess_dyn.run(None, {'X': x_i})[0]
    diffs_dyn.append(np.mean(np.abs(y_fp - y_dy)))
    if HAS_STATIC:
        y_st = sess_stat.run(None, {'X': x_i})[0]
        diffs_stat.append(np.mean(np.abs(y_fp - y_st)))

diffs_dyn = np.array(diffs_dyn)

print(f"Accuracy analysis over {n_samples} random inputs:")
print(f"\nDynamic INT8 vs FP32:")
print(f"  Mean abs error: {diffs_dyn.mean():.6f}")
print(f"  Std abs error:  {diffs_dyn.std():.6f}")
print(f"  P95 abs error:  {np.percentile(diffs_dyn, 95):.6f}")
print(f"  Max abs error:  {diffs_dyn.max():.6f}")

if HAS_STATIC:
    diffs_stat = np.array(diffs_stat)
    print(f"\nStatic INT8 vs FP32:")
    print(f"  Mean abs error: {diffs_stat.mean():.6f}")
    print(f"  Std abs error:  {diffs_stat.std():.6f}")
    print(f"  P95 abs error:  {np.percentile(diffs_stat, 95):.6f}")
    print(f"  Max abs error:  {diffs_stat.max():.6f}")

---
## Exercise 6: Performance Benchmark (FP32 vs Quantized)

INT8 computation can be 2-4x faster than FP32 on CPUs with VNNI/AVX-512 support.
Even without hardware INT8, reduced memory bandwidth helps.

**Throughput** is computed as:

$$\text{Throughput} = \frac{N_{\text{samples}}}{T_{\text{total}}} \text{ samples/sec}$$

In [ ]:
n_warmup = 100
n_runs = 2000
x_bench = np.random.randn(N, D_in).astype(np.float32)

def benchmark_session(sess, input_name, x, n_warmup, n_runs):
    for _ in range(n_warmup):
        sess.run(None, {input_name: x})
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {input_name: x})
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

t_fp32 = benchmark_session(sess_fp32, 'X', x_bench, n_warmup, n_runs)
t_dyn  = benchmark_session(sess_dyn, 'X', x_bench, n_warmup, n_runs)

print(f"{'Metric':<20} {'FP32':>12} {'Dynamic INT8':>14}")
print('-' * 50)
print(f"{'Mean (ms)':<20} {t_fp32.mean():>12.4f} {t_dyn.mean():>14.4f}")
print(f"{'P50 (ms)':<20} {np.percentile(t_fp32,50):>12.4f} {np.percentile(t_dyn,50):>14.4f}")
print(f"{'P95 (ms)':<20} {np.percentile(t_fp32,95):>12.4f} {np.percentile(t_dyn,95):>14.4f}")
print(f"{'P99 (ms)':<20} {np.percentile(t_fp32,99):>12.4f} {np.percentile(t_dyn,99):>14.4f}")

throughput_fp32 = N * 1000.0 / t_fp32.mean()
throughput_dyn  = N * 1000.0 / t_dyn.mean()
print(f"{'Throughput (samp/s)':<20} {throughput_fp32:>12.0f} {throughput_dyn:>14.0f}")
print(f"{'Speedup':<20} {'1.00x':>12} {t_fp32.mean()/t_dyn.mean():>13.2f}x")

if HAS_STATIC:
    t_stat = benchmark_session(sess_stat, 'X', x_bench, n_warmup, n_runs)
    print(f"\nStatic INT8:")
    print(f"  Mean: {t_stat.mean():.4f} ms, Speedup: {t_fp32.mean()/t_stat.mean():.2f}x")

---
## Exercise 7: Inspect Quantized Model Structure

Quantized models insert QuantizeLinear (Q) and DequantizeLinear (DQ) nodes.
In QDQ format:
```
FP32_input → Q → DQ → INT8_op → Q → DQ → FP32_output
```
The runtime recognizes these patterns and executes with INT8 kernels.

In [ ]:
from collections import Counter

print("=" * 60)
print("FP32 Model Structure")
print("=" * 60)
for i, n in enumerate(fp32_model.graph.node):
    print(f"  [{i}] {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

print(f"\n{'=' * 60}")
print("Dynamic INT8 Model Structure")
print("=" * 60)
dyn_m = onnx.load(dynamic_path)
for i, n in enumerate(dyn_m.graph.node):
    print(f"  [{i}] {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

print(f"\nOp histogram:")
for op, cnt in Counter(n.op_type for n in dyn_m.graph.node).most_common():
    print(f"  {op}: {cnt}")

# Check quantized weight data types
print(f"\nInitializer dtypes:")
for init in dyn_m.graph.initializer:
    arr = numpy_helper.to_array(init)
    print(f"  {init.name:<20} dtype={arr.dtype}, shape={arr.shape}")
    if arr.dtype == np.int8:
        print(f"    range: [{arr.min()}, {arr.max()}]  (INT8: [-128, 127])")

---
## Exercise 8: Manual Quantization Math Verification

Verify the quantization formula by hand on a small weight tensor.

For symmetric INT8 quantization:

$$s = \frac{\max(|W|)}{127}, \quad W_q = \text{clamp}\!\left(\left\lfloor \frac{W}{s} \right\rceil, -128, 127\right)$$

$$\hat{W} = s \cdot W_q, \quad \text{error} = |W - \hat{W}| \leq \frac{s}{2}$$

In [ ]:
# Manual quantization of a small tensor
w_small = np.array([
    [0.15, -0.23, 0.08, -0.41],
    [0.32, -0.05, 0.19, -0.28],
    [0.01, -0.37, 0.44, -0.12],
], dtype=np.float32)

# Symmetric quantization (zero-point = 0)
abs_max = np.max(np.abs(w_small))
scale = abs_max / 127.0
w_quantized = np.clip(np.round(w_small / scale), -128, 127).astype(np.int8)
w_dequantized = scale * w_quantized.astype(np.float32)
error = np.abs(w_small - w_dequantized)

print("Manual Symmetric INT8 Quantization")
print("=" * 50)
print(f"\nOriginal W (FP32):")
print(w_small)
print(f"\nabs_max = {abs_max:.4f}")
print(f"scale s = abs_max / 127 = {scale:.6f}")
print(f"\nQuantized W_q (INT8):")
print(w_quantized)
print(f"\nDequantized W_hat = s * W_q:")
print(w_dequantized)
print(f"\nQuantization error |W - W_hat|:")
print(error)

theoretical_max_error = scale / 2
actual_max_error = error.max()
print(f"\nTheoretical max error (s/2): {theoretical_max_error:.6f}")
print(f"Actual max error:            {actual_max_error:.6f}")
assert actual_max_error <= theoretical_max_error + 1e-7, "Error exceeds bound!"
print(f"Error within bound: True")

# Information loss
snr = 10 * np.log10(np.mean(w_small**2) / np.mean(error**2 + 1e-15))
print(f"\nSignal-to-Noise Ratio: {snr:.1f} dB")

---
## Challenge: Quantize and Benchmark a Multi-Layer Model

Build a deeper MLP, quantize it with both dynamic and static methods, and produce
a comprehensive comparison report covering size, speed, and accuracy.

In [ ]:
def build_deep_mlp(n_layers=4, d_in=128, d_hidden=256, d_out=64, batch=16):
    """Build a deeper MLP for quantization benchmarking."""
    np.random.seed(0)
    inits, nodes = [], []
    prev_d, prev_name = d_in, 'X'

    for i in range(n_layers):
        d_next = d_hidden if i < n_layers - 1 else d_out
        W = np.random.randn(prev_d, d_next).astype(np.float32) * np.sqrt(2.0/prev_d)
        b = np.zeros(d_next, dtype=np.float32)
        inits.append(numpy_helper.from_array(W, f'W{i}'))
        inits.append(numpy_helper.from_array(b, f'b{i}'))
        nodes.append(helper.make_node('MatMul', [prev_name, f'W{i}'], [f'h{i}']))
        nodes.append(helper.make_node('Add', [f'h{i}', f'b{i}'], [f'a{i}']))
        if i < n_layers - 1:
            nodes.append(helper.make_node('Relu', [f'a{i}'], [f'r{i}']))
            prev_name = f'r{i}'
        else:
            prev_name = f'a{i}'
        prev_d = d_next

    X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch, d_in])
    Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch, d_out])
    g = helper.make_graph(nodes, 'deep_mlp', [X_i], [Y_i], initializer=inits)
    return helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

deep_model = build_deep_mlp(n_layers=5, d_in=256, d_hidden=512, d_out=128, batch=32)
check_model(deep_model)

deep_fp32_path = os.path.join(TMPDIR, 'deep_fp32.onnx')
deep_dyn_path  = os.path.join(TMPDIR, 'deep_dyn.onnx')
onnx.save(deep_model, deep_fp32_path)

quantize_dynamic(deep_fp32_path, deep_dyn_path, weight_type=QuantType.QInt8)

# Benchmark
sess_fp = ort.InferenceSession(deep_fp32_path, providers=['CPUExecutionProvider'])
sess_dy = ort.InferenceSession(deep_dyn_path, providers=['CPUExecutionProvider'])

x_b = np.random.randn(32, 256).astype(np.float32)
n_w, n_r = 50, 1000

t_fp = benchmark_session(sess_fp, 'X', x_b, n_w, n_r)
t_dy = benchmark_session(sess_dy, 'X', x_b, n_w, n_r)

y_ref = sess_fp.run(None, {'X': x_b})[0]
y_dq  = sess_dy.run(None, {'X': x_b})[0]

print("Deep MLP Quantization Report")
print("=" * 55)
print(f"  Layers: 5, Dims: 256 -> 512 -> 512 -> 512 -> 512 -> 128")
print(f"  Batch size: 32")
print(f"\n{'Metric':<22} {'FP32':>10} {'INT8 Dyn':>10}")
print('-' * 45)
print(f"{'File size (KB)':<22} {os.path.getsize(deep_fp32_path)/1024:>10.1f} {os.path.getsize(deep_dyn_path)/1024:>10.1f}")
print(f"{'Mean latency (ms)':<22} {t_fp.mean():>10.4f} {t_dy.mean():>10.4f}")
print(f"{'P99 latency (ms)':<22} {np.percentile(t_fp,99):>10.4f} {np.percentile(t_dy,99):>10.4f}")
print(f"{'Throughput (samp/s)':<22} {32*1000/t_fp.mean():>10.0f} {32*1000/t_dy.mean():>10.0f}")
print(f"{'Max |diff|':<22} {'--':>10} {np.max(np.abs(y_ref-y_dq)):>10.6f}")
print(f"{'Speedup':<22} {'1.00x':>10} {t_fp.mean()/t_dy.mean():>9.2f}x")
print(f"{'Compression':<22} {'1.00x':>10} {os.path.getsize(deep_fp32_path)/os.path.getsize(deep_dyn_path):>9.2f}x")

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Dynamic quantization | `quantize_dynamic` — no calibration needed, weights quantized offline |
| Static quantization | `quantize_static` + `CalibrationDataReader` — activations calibrated offline |
| Quantization math | $x_q = \text{clamp}(\lfloor x/s \rceil + z)$, error $\leq s/2$ |
| Size reduction | ~2-4x compression from FP32 to INT8 |
| Accuracy impact | Measured mean/max absolute error across many inputs |
| Performance | Benchmarked latency and throughput improvements |
| QDQ format | QuantizeLinear/DequantizeLinear node insertion pattern |

**Next:** [Pruning and Sparsity](../03_Pruning_and_Sparsity/)